In [2]:
import os
import random
import shutil
from tqdm import tqdm

# ------------------- PATHS ------------------- #
IMAGE_DIR = "/kaggle/input/datasets/uzair2000/for-diffusion/hello/train/images"
LABEL_DIR = "/kaggle/input/datasets/uzair2000/for-diffusion/hello/train/labels"
OUTPUT_DIR = "/kaggle/working/diffusion_dataset_augmented"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------- Caption Components ------------------- #
camera_styles = [
    "grainy CCTV surveillance footage",
    "low resolution security camera footage",
    "blurry surveillance camera footage",
    "noisy CCTV security camera footage",
    "overhead security camera footage",
    "wide angle CCTV surveillance footage"
]

camera_angles = [
    "captured from overhead angle",
    "captured from side angle",
    "captured from high angle",
    "captured from wide angle"
]

times = [
    "at night",
    "during daytime",
    "in the evening",
    "early morning"
]

lighting = [
    "under dim lighting",
    "under bright street lights",
    "in poor lighting conditions",
    "with strong shadows"
]

locations = [
    "in a street",
    "inside a small shop",
    "in a parking lot",
    "inside a building hallway",
    "near a building entrance",
    "in a train station",
    "in a narrow alley"
]

environment_details = [
    "near parked cars",
    "near a shop entrance",
    "near a staircase",
    "near a security gate",
    "near trash bins",
    "in an empty corridor"
]

violence_actions = [
    "punching each other",
    "kicking and hitting each other",
    "pushing and shoving violently",
    "snatching a bag from another person",
    "attacking another individual aggressively"
]

weapon_actions = [
    "holding a knife",
    "holding a handgun",
    "holding a rifle",
    "threatening others with a weapon",
    "walking while carrying a weapon"
]

background_actions = [
    "showing an empty quiet scene",
    "capturing a normal environment with no suspicious activity",
    "showing a peaceful surveillance scene"
]

# ------------------- Caption Generator ------------------- #
def generate_captions(label_file, num_captions=50):
    """
    Generate multiple captions for one image based on YOLO labels
    """
    captions = []

    # Read YOLO label
    if os.path.exists(label_file):
        with open(label_file, "r") as f:
            lines = f.readlines()
    else:
        lines = []

    num_objects = len(lines)
    class_ids = [int(line.split()[0]) for line in lines] if num_objects > 0 else []

    for _ in range(num_captions):
        camera = random.choice(camera_styles)
        angle = random.choice(camera_angles)
        time = random.choice(times)
        light = random.choice(lighting)
        location = random.choice(locations)
        env = random.choice(environment_details)

        if len(lines) == 0:
            # Background image
            action = random.choice(background_actions)
        else:
            if 0 in class_ids:
                act = random.choice(violence_actions)
                action = f"showing {num_objects} people {act}" if num_objects > 1 else f"showing one person {act}"
            elif 1 in class_ids:
                act = random.choice(weapon_actions)
                action = f"showing {num_objects} people {act}" if num_objects > 1 else f"showing one person {act}"
            else:
                action = random.choice(background_actions)

        caption = f"{camera} {angle} {time} {light}, {action} {location} {env}."
        captions.append(caption)

    return captions

# ------------------- Processing Dataset ------------------- #
images = os.listdir(IMAGE_DIR)

for img_name in tqdm(images):
    img_path = os.path.join(IMAGE_DIR, img_name)
    label_name = img_name.replace(".jpg", ".txt").replace(".png", ".txt")
    label_path = os.path.join(LABEL_DIR, label_name)

    # Generate captions (single file)
    captions = generate_captions(label_path, num_captions=50)

    # Save image (only once)
    img_out_file = os.path.join(OUTPUT_DIR, img_name)
    if not os.path.exists(img_out_file):
        shutil.copy(img_path, img_out_file)

    # Save all captions in one .txt file
    caption_file = os.path.join(OUTPUT_DIR, f"{os.path.splitext(img_name)[0]}.txt")
    with open(caption_file, "w") as f:
        for idx, cap in enumerate(captions, 1):
            f.write(f"Caption {idx}: {cap}\n")

print("Augmented dataset creation complete!")
print("Saved at:", OUTPUT_DIR)

100%|██████████| 10816/10816 [01:18<00:00, 137.11it/s]

Augmented dataset creation complete!
Saved at: /kaggle/working/diffusion_dataset_augmented


In [3]:
import shutil
import os

# Folder to zip
folder_to_zip = "/kaggle/working/diffusion_dataset_augmented"

# Output zip file path (Kaggle output)
zip_output_path = "/kaggle/working/diffusion_dataset_augmented"

# Create zip (this will produce diffusion_dataset_augmented.zip)
shutil.make_archive(zip_output_path, 'zip', folder_to_zip)

print("Folder zipped successfully!")
print("Saved as:", zip_output_path + ".zip")

Folder zipped successfully!
Saved as: /kaggle/working/diffusion_dataset_augmented.zip
